In [4]:
import os, re, json, pandas as pd

root = r"J:\All Data"
query = "JPM Bonds Data.csv"
regex = False
ignore_case = True
include_markdown = False
include_outputs = False
relative_paths = True

IGNORE_DIRS = {".ipynb_checkpoints", ".git", ".svn", ".hg", ".venv", "venv", "__pycache__"}

def compile_pattern(q, is_regex, ic):
    flags = re.IGNORECASE if ic else 0
    return re.compile(q, flags) if is_regex else re.compile(re.escape(q), flags)

def lines_from_source(src):
    if src is None: return []
    if isinstance(src, list): return [s.rstrip("\n") for s in src]
    if isinstance(src, str): return src.splitlines()
    return []

def iter_notebooks(root_path):
    for base, dirs, files in os.walk(root_path):
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]
        for f in files:
            if f.endswith(".ipynb"):
                yield os.path.join(base, f)

def search_cell(cell, pattern, include_md, include_out):
    hits = []
    ct = cell.get("cell_type")
    if ct == "code":
        src = lines_from_source(cell.get("source"))
        for i, line in enumerate(src, 1):
            if pattern.search(line):
                hits.append(("code", i, line))
        if include_out:
            outs = cell.get("outputs") or []
            for out in outs:
                if "text" in out:
                    for i, line in enumerate(lines_from_source(out.get("text")), 1):
                        if pattern.search(line):
                            hits.append(("output", i, line))
                if "data" in out:
                    data = out.get("data") or {}
                    for mime, payload in data.items():
                        if isinstance(payload, str):
                            out_lines = payload.splitlines()
                        elif isinstance(payload, list):
                            out_lines = [str(x) for x in payload]
                        else:
                            out_lines = [json.dumps(payload, ensure_ascii=False)]
                        for i, line in enumerate(out_lines, 1):
                            if pattern.search(line):
                                hits.append((f"output:{mime}", i, line))
    elif ct == "markdown" and include_md:
        src = lines_from_source(cell.get("source"))
        for i, line in enumerate(src, 1):
            if pattern.search(line):
                hits.append(("markdown", i, line))
    return hits

def search_notebook(path, pattern, include_md, include_out):
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except Exception as e:
        return [], str(e)
    cells = nb.get("cells") or []
    results = []
    for idx, cell in enumerate(cells):
        hits = search_cell(cell, pattern, include_md, include_out)
        for kind, line_no, line in hits:
            results.append({"file": path, "cell_index": idx, "cell_type": kind, "line_no": line_no, "line": line})
    return results, None

if not os.path.exists(root):
    raise FileNotFoundError(f"Path not found: {root}")

pattern = compile_pattern(query, regex, ignore_case)
all_results, errors = [], []

for nb_path in iter_notebooks(root):
    res, err = search_notebook(nb_path, pattern, include_markdown, include_outputs)
    if err:
        errors.append({"file": nb_path, "error": err})
    all_results.extend(res)

if relative_paths:
    for r in all_results:
        r["file"] = os.path.relpath(r["file"], root)

df = pd.DataFrame(all_results, columns=["file","cell_index","cell_type","line_no","line"])

if df.empty:
    print("No matches found.")
else:
    for r in all_results:
        print(f'{r["file"]}: cell {r["cell_index"]} [{r["cell_type"]}] line {r["line_no"]}: {r["line"]}')
    print(f"\nTotal matches: {len(all_results)}")
    print(f"Files matched: {df['file'].nunique()}")

if errors:
    print("\nErrors:")
    for e in errors:
        print(f'{e["file"]}: {e["error"]}')

df

53A. HY Bonds Basis Email\Untitled.ipynb: cell 0 [code] line 4: query = "JPM Bonds Data.csv"
53A. HY Bonds Basis Email\Old\test.ipynb: cell 17 [code] line 1: df = pd.read_csv("JPM Bonds Data.csv",index_col=0,parse_dates=True).iloc[:-2,:]

Total matches: 2
Files matched: 2


,file,cell_index,cell_type,line_no,line
0,53A. HY Bonds Basis Email\Untitled.ipynb,0,code,4,"query = ""JPM Bonds Data.csv"""
1,53A. HY Bonds Basis Email\Old\test.ipynb,17,code,1,"df = pd.read_csv(""JPM Bonds Data.csv"",index_co..."


In [3]:
df = pd.read_excel("JPM Bonds Data.xlsx")
df

FileNotFoundError: [Errno 2] No such file or directory: 'JPM Bonds Data.xlsx'